# D3b: Contrastive Learning

In [ ]:
# ── Setup: clone repo + install deps ─────────────────────────
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
!pip install hdbscan umap-learn -q

import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')


In [ ]:
# ── Imports ──────────────────────────────────────────────────
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, adjusted_rand_score,
    normalized_mutual_info_score, f1_score, classification_report, confusion_matrix)
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')


In [ ]:
# ── Cycle extraction (matching Helene's tsne_v2.py) ────────
# Key: double savgol, 300 deg/s min peak height, resample boundaries,
# overlap-based pairing, omega in rad/s in cycle matrix

CONFIG = {
    'FS': 50.0,
    'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21,
    'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0,   # strict: reject weak peaks
    'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0,
    'TARGET_LEN': 64,
    'MIN_CYCLE_SAMPLES': 10,
}

KNOWN_PATTERNS = {'overhand', 'sneak_overhand', 'underhand', 'sneak_underhand',
                  'dragon_roll', 'underhand_default'}
EXCLUDE_LABELS = {None, 'CW', 'CCW'}  # never use CW/CCW as classes

_EXACT = {'underhand':'underhand','overhand':'overhand','dragon_roll':'dragon_roll',
          'sneak_underhand':'sneak_underhand','sneak_overhand':'sneak_overhand','idle':None}
_PRULES = [
    (re.compile(r'^us', re.I), 'sneak_underhand'), (re.compile(r'^os', re.I), 'sneak_overhand'),
    (re.compile(r'^u', re.I), 'underhand'), (re.compile(r'^o', re.I), 'overhand'),
    (re.compile(r'^fb', re.I), 'dragon_roll'), (re.compile(r'^bf', re.I), 'dragon_roll'),
    (re.compile(r'^cw$', re.I), 'CW'), (re.compile(r'^ccw$', re.I), 'CCW'),
    (re.compile(r'^idle', re.I), None), (re.compile(r'^vq', re.I), None),
]
def map_label(raw):
    raw = raw.strip()
    if raw.lower() in _EXACT: return _EXACT[raw.lower()]
    for pat, c in _PRULES:
        if pat.match(raw): return c
    return None

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w','ay_w','az_w']].values
    omega = df[['gx','gy','gz']].values * (np.pi / 180.0)
    return t, A, omega

def detect_cycles(t, omega, fs=50.0):
    mag_deg = np.linalg.norm(omega, axis=1) * (180.0 / np.pi)
    n = len(mag_deg)
    if n < 7: return [], mag_deg, np.array([], dtype=int)
    win = int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if win % 2 == 0: win += 1
    win = max(5, min(win, n if n % 2 == 1 else n - 1))
    poly = max(1, min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']), win - 2))
    mag_s = savgol_filter(mag_deg, window_length=win, polyorder=poly, mode='interp')
    mag_s = savgol_filter(mag_s, window_length=win, polyorder=poly, mode='interp')
    peaks, _ = find_peaks(mag_s, distance=max(1, int(CONFIG['CYCLE_MIN_PERIOD_S'] * fs)),
                          prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],
                          height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(peaks) == 0: return [], mag_s, peaks
    cycles = []
    for i, p in enumerate(peaks):
        left = 0 if i == 0 else (peaks[i-1] + p) // 2
        right = (len(t)-1) if i == len(peaks)-1 else (p + peaks[i+1]) // 2
        if right > left and (right - left) >= CONFIG['MIN_CYCLE_SAMPLES']:
            cycles.append((left, right))
    return cycles, mag_s, peaks

def pair_cycles(t0, cyc0, t1, cyc1):
    paired0, paired1, used = [], [], set()
    for c0 in cyc0:
        best_i, best_ov = -1, -1.0
        for i, c1 in enumerate(cyc1):
            if i in used: continue
            ov = max(0, min(t0[c0[1]], t1[c1[1]]) - max(t0[c0[0]], t1[c1[0]]))
            if ov > best_ov: best_ov, best_i = ov, i
        if best_i >= 0 and best_ov > 0:
            paired0.append(c0); paired1.append(cyc1[best_i]); used.add(best_i)
    return paired0, paired1

def resample(sig, n):
    if len(sig) < 2:
        return np.zeros(n) if sig.ndim == 1 else np.zeros((n, sig.shape[1]))
    x_old, x_new = np.linspace(0, 1, len(sig)), np.linspace(0, 1, n)
    if sig.ndim == 1: return np.interp(x_new, x_old, sig)
    return np.column_stack([np.interp(x_new, x_old, sig[:, j]) for j in range(sig.shape[1])])

def build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1):
    tl = CONFIG['TARGET_LEN']
    r0 = resample(np.column_stack([A0[s0:e0], om0[s0:e0]]), tl)
    r1 = resample(np.column_stack([A1[s1:e1], om1[s1:e1]]), tl)
    return np.column_stack([r0, r1]).T.astype(np.float32)  # (12, 64)

print('Functions defined')


In [ ]:
# ── Load all cycles + labels ─────────────────────────────────

all_matrices = []; all_labels = []; all_groups = []; session_names = []

def load_session_cycles(d0_path, d1_path, name, label='unlabeled', windows=None):
    df0, df1 = pd.read_csv(d0_path), pd.read_csv(d1_path)
    t0, A0, om0 = extract_signals(df0)
    t1, A1, om1 = extract_signals(df1)
    cyc0, _, _ = detect_cycles(t0, om0, CONFIG['FS'])
    cyc1, _, _ = detect_cycles(t1, om1, CONFIG['FS'])
    p0, p1 = pair_cycles(t0, cyc0, t1, cyc1)
    if windows is not None:
        fp0, fp1 = [], []
        for (s0,e0),(s1,e1) in zip(p0,p1):
            t_mid = (t0[s0]+t0[e0])/2
            if any(ws <= t_mid < we for ws, we in windows):
                fp0.append((s0,e0)); fp1.append((s1,e1))
        p0, p1 = fp0, fp1
    grp = name.split('/')[0] if '/' in name else name
    count = 0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1)
        all_matrices.append(cm); all_labels.append(label); all_groups.append(grp)
        count += 1
    if count > 0: session_names.append(name)
    return count

# Homogeneous
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED, '*_device0_processed.csv'))):
    d1 = d0.replace('_device0_', '_device1_')
    if not os.path.exists(d1): continue
    stem = os.path.basename(d0).replace('_device0_processed.csv', '')
    parts = stem.split('_')
    if len(parts) < 3: continue
    rest = '_'.join(parts[2:])
    label = 'unlabeled'
    for pat in sorted(KNOWN_PATTERNS, key=len, reverse=True):
        if rest.startswith(pat):
            label = 'underhand' if pat == 'underhand_default' else pat; break
    n = load_session_cycles(d0, d1, stem, label)
    if n > 0: print(f'  {stem:<50s} {label:<20s} {n:>4d}')

# Heterogeneous
if os.path.isdir(NEW_LABELED_RAW):
    for sname in sorted(os.listdir(NEW_LABELED_RAW)):
        sdir = os.path.join(NEW_LABELED_RAW, sname)
        if not os.path.isdir(sdir): continue
        lpath = None
        for fn in ('labels_corrected.json', 'labels.json'):
            p = os.path.join(sdir, fn)
            if os.path.isfile(p): lpath = p; break
        if not lpath: continue
        d0 = os.path.join(DATA_PROCESSED, sname + '_device0_processed.csv')
        d1 = os.path.join(DATA_PROCESSED, sname + '_device1_processed.csv')
        if not (os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lpath, encoding='utf-8') as f: data = _json.load(f)
        segs = data.get('segments', data) if isinstance(data, dict) else data
        wbl = defaultdict(list)
        for seg in segs:
            canon = map_label(seg.get('label', ''))
            if canon in EXCLUDE_LABELS: continue
            s, e = seg.get('start'), seg.get('end')
            if s is None: continue
            if e is None: e = s + 2.0
            wbl[canon].append((float(s), float(e)))
        for label, windows in sorted(wbl.items()):
            n = load_session_cycles(d0, d1, sname + '/' + label, label, windows)
            if n > 0: print(f'  {sname}/{label:<20s} {"":>29s} {n:>4d}')

X_raw = np.array([cm.ravel() for cm in all_matrices])  # (N, 768)
y_all = np.array(all_labels)
g_all = np.array(all_groups)

# Masks
labeled_mask = np.array([l not in ('unlabeled', 'CW', 'CCW') for l in y_all])
X_labeled = X_raw[labeled_mask]
y_labeled = y_all[labeled_mask]
g_labeled = g_all[labeled_mask]

print('\nTotal: ' + str(len(X_raw)) + ' cycles (' + str(X_raw.shape[1]) + 'D)')
print('Labeled: ' + str(labeled_mask.sum()) + ', Unlabeled: ' + str((~labeled_mask).sum()))
print('Label distribution:')
for lab, cnt in sorted(Counter(y_labeled).items(), key=lambda x: -x[1]):
    print(f'  {lab:<20s}: {cnt:>5d}')
print('Groups: ' + str(len(np.unique(g_labeled))))


In [ ]:
# ── PCA + StandardScaler ──────────────────────────────────────

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_labeled_scaled = X_scaled[labeled_mask]

pca_full = PCA(n_components=min(50, X_scaled.shape[1], X_scaled.shape[0]-1), svd_solver='full')
X_pca = pca_full.fit_transform(X_scaled)
X_labeled_pca = X_pca[labeled_mask]
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
print('PCA: 768D -> ' + str(X_pca.shape[1]) + 'D (' + str(round(cumvar[-1]*100,1)) + '% var)')


In [ ]:
# ── LOSO helper ──────────────────────────────────────────────

def run_loso(X, y, groups, make_clf, preprocess=None):
    """Generic LOSO. make_clf() returns a fresh classifier. preprocess(X_tr, X_te) returns transformed pair."""
    unique_groups = np.unique(groups)
    # Find singleton classes (only 1 group)
    class_groups = defaultdict(set)
    for label, grp in zip(y, groups): class_groups[label].add(grp)
    singleton_classes = {c for c, gs in class_groups.items() if len(gs) < 2}
    test_groups = [g for g in unique_groups if not all(y[groups == g][0] in singleton_classes for _ in [1])]
    # Actually: skip groups where ALL their labels are singletons
    test_groups = []
    for g in unique_groups:
        g_labels = set(y[groups == g])
        if not g_labels.issubset(singleton_classes):
            test_groups.append(g)
    
    all_true, all_pred = [], []
    for g in test_groups:
        te = groups == g; tr = ~te
        X_tr, X_te, y_tr, y_te = X[tr], X[te], y[tr], y[te]
        if preprocess:
            X_tr, X_te = preprocess(X_tr, X_te)
        clf = make_clf()
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_te)
        all_true.extend(y_te.tolist()); all_pred.extend(preds.tolist())
    
    all_true, all_pred = np.array(all_true), np.array(all_pred)
    labels = sorted(set(all_true) | set(all_pred))
    f1_mac = f1_score(all_true, all_pred, average='macro', labels=labels, zero_division=0)
    f1_wt = f1_score(all_true, all_pred, average='weighted', labels=labels, zero_division=0)
    cm = confusion_matrix(all_true, all_pred, labels=labels)
    report = classification_report(all_true, all_pred, labels=labels, zero_division=0)
    acc = np.mean(all_true == all_pred)
    return {'f1_macro': f1_mac, 'f1_weighted': f1_wt, 'accuracy': acc,
            'cm': cm, 'labels': labels, 'report': report,
            'all_true': all_true, 'all_pred': all_pred}

def print_results(res, name):
    print('\n' + '='*60)
    print(name)
    print('='*60)
    print(res['report'])
    print(f'Accuracy: {res["accuracy"]:.3f}')
    print(f'F1 macro: {res["f1_macro"]:.3f}')
    print(f'F1 weighted: {res["f1_weighted"]:.3f}')

def plot_cm(cm, labels, title, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()*0.5 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.colorbar(im, ax=ax); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150)
    plt.show()

print('LOSO helper ready')


## D3b: Contrastive Learning

Learn an embedding where same-movement cycles are close and different ones are far.
Then classify in embedding space with k-NN or linear probe.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# ── Encoder network ─────────────────────────────────────────
class CycleEncoder(nn.Module):
    def __init__(self, in_channels=12, seq_len=64, embed_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj = nn.Sequential(
            nn.Linear(128, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim)
        )
    def forward(self, x):
        h = self.conv(x).squeeze(-1)  # (B, 128)
        return F.normalize(self.proj(h), dim=1)  # (B, embed_dim) unit sphere

# ── Supervised contrastive loss ──────────────────────────────
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.t = temperature
    def forward(self, embeddings, labels):
        # embeddings: (B, D), labels: (B,)
        sim = torch.mm(embeddings, embeddings.T) / self.t  # (B, B)
        mask_pos = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        mask_self = torch.eye(len(labels), device=sim.device)
        mask_pos = mask_pos - mask_self  # exclude self
        exp_sim = torch.exp(sim - sim.max(dim=1, keepdim=True).values)  # stability
        exp_sim = exp_sim * (1 - mask_self)  # exclude self from denominator
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        loss = -(mask_pos * log_prob).sum(dim=1) / (mask_pos.sum(dim=1) + 1e-8)
        return loss.mean()

# ── Augmentation ─────────────────────────────────────────────
def augment(x, shift_max=4, noise_std=0.02):
    # Random circular shift
    shift = np.random.randint(-shift_max, shift_max+1)
    x = np.roll(x, shift, axis=1)
    # Add noise
    x = x + np.random.randn(*x.shape).astype(np.float32) * noise_std
    return x

class CycleDataset(Dataset):
    def __init__(self, X, y, aug=False):
        self.X = X.reshape(-1, 12, 64).astype(np.float32)
        self.y = y
        self.le = {l: i for i, l in enumerate(sorted(set(y)))}
        self.aug = aug
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].copy()
        if self.aug: x = augment(x)
        return torch.from_numpy(x), self.le[self.y[i]]

print('Model and loss defined')


In [ ]:
# ── Train contrastive encoder on labeled data ────────────────
EMBED_DIM = 64
EPOCHS = 30
BATCH_SIZE = 128
LR = 1e-3

ds = CycleDataset(X_labeled, y_labeled, aug=True)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

encoder = CycleEncoder(embed_dim=EMBED_DIM).to(device)
criterion = SupConLoss(temperature=0.1)
optimizer = torch.optim.Adam(encoder.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'Training contrastive encoder ({EPOCHS} epochs, {len(ds)} samples)...')
losses = []
for epoch in range(EPOCHS):
    encoder.train()
    epoch_loss = 0; n_batch = 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        emb = encoder(xb)
        loss = criterion(emb, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item(); n_batch += 1
    scheduler.step()
    avg_loss = epoch_loss / max(n_batch, 1)
    losses.append(avg_loss)
    if (epoch+1) % 5 == 0:
        print(f'  Epoch {epoch+1}/{EPOCHS}: loss={avg_loss:.4f}')

plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('SupCon Loss'); plt.title('Training'); plt.show()


In [ ]:
# ── Extract embeddings ──────────────────────────────────────
encoder.eval()
with torch.no_grad():
    X_tensor = torch.from_numpy(X_labeled.reshape(-1, 12, 64).astype(np.float32)).to(device)
    # Batch to avoid OOM
    embs = []
    for i in range(0, len(X_tensor), 256):
        embs.append(encoder(X_tensor[i:i+256]).cpu().numpy())
    X_embed = np.vstack(embs)
print(f'Embeddings: {X_embed.shape}')

# Visualize with t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne_embed = tsne.fit_transform(X_embed)
COLORS = {'underhand':'#5DCAA5','overhand':'#7F77DD','dragon_roll':'#E24B4A',
          'sneak_overhand':'#D85A30','sneak_underhand':'#EF9F27'}
fig, ax = plt.subplots(figsize=(10, 8))
for lab in sorted(set(y_labeled)):
    m = y_labeled == lab
    ax.scatter(X_tsne_embed[m, 0], X_tsne_embed[m, 1], c=COLORS.get(lab, '#888'),
               s=12, alpha=0.7, label=lab)
ax.legend(); ax.set_title('Contrastive embeddings (t-SNE)'); plt.tight_layout(); plt.show()


In [ ]:
# ── LOSO: k-NN on contrastive embeddings ────────────────────
# Re-extract embeddings per fold (train encoder on train, embed test)
unique_groups = np.unique(g_labeled)
class_groups = defaultdict(set)
for label, grp in zip(y_labeled, g_labeled): class_groups[label].add(grp)
singleton_classes = {c for c, gs in class_groups.items() if len(gs) < 2}
test_groups = [g for g in unique_groups if not set(y_labeled[g_labeled == g]).issubset(singleton_classes)]

all_true, all_pred = [], []
print('LOSO with contrastive embeddings + k-NN...')
for gi, g in enumerate(test_groups):
    te = g_labeled == g; tr = ~te
    X_tr = X_labeled[tr].reshape(-1, 12, 64).astype(np.float32)
    X_te = X_labeled[te].reshape(-1, 12, 64).astype(np.float32)
    y_tr, y_te = y_labeled[tr], y_labeled[te]
    # Train fresh encoder
    enc = CycleEncoder(embed_dim=EMBED_DIM).to(device)
    ds_tr = CycleDataset(X_tr.reshape(-1, 768), y_tr, aug=True)
    dl_tr = DataLoader(ds_tr, batch_size=min(128, len(ds_tr)), shuffle=True, drop_last=len(ds_tr)>128)
    opt = torch.optim.Adam(enc.parameters(), lr=LR, weight_decay=1e-4)
    crit = SupConLoss(temperature=0.1)
    enc.train()
    for ep in range(15):  # fewer epochs per fold for speed
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            loss = crit(enc(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    # Embed
    enc.eval()
    with torch.no_grad():
        emb_tr = enc(torch.from_numpy(X_tr).to(device)).cpu().numpy()
        emb_te = enc(torch.from_numpy(X_te).to(device)).cpu().numpy()
    # k-NN
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn.fit(emb_tr, y_tr)
    preds = knn.predict(emb_te)
    all_true.extend(y_te.tolist()); all_pred.extend(preds.tolist())
    if (gi+1) % 5 == 0: print(f'  fold {gi+1}/{len(test_groups)}')

all_true, all_pred = np.array(all_true), np.array(all_pred)
labels = sorted(set(all_true) | set(all_pred))
f1_con = f1_score(all_true, all_pred, average='macro', labels=labels, zero_division=0)
acc_con = np.mean(all_true == all_pred)
cm_con = confusion_matrix(all_true, all_pred, labels=labels)
report_con = classification_report(all_true, all_pred, labels=labels, zero_division=0)
print('\n' + report_con)
print(f'F1 macro: {f1_con:.3f}, Acc: {acc_con:.3f}')
plot_cm(cm_con, labels, f'Contrastive + k-NN (F1={f1_con:.3f})')


In [ ]:
print('='*60)
print('D3b: CONTRASTIVE LEARNING SUMMARY')
print('='*60)
print(f'Labeled cycles: {len(X_labeled)}, Embedding dim: {EMBED_DIM}')
print(f'Encoder: 3-layer 1D CNN, SupCon loss, temp=0.1')
print(f'LOSO classifier: k-NN (k=5, cosine distance)')
print(f'Training: {EPOCHS} epochs global, 15 epochs per LOSO fold')
print(f'')
print(f'Contrastive + k-NN:  F1 macro={f1_con:.3f}, Acc={acc_con:.3f}')
print(f'V08 reference:       F1 macro=0.632')
print(f'Delta vs V08:        {f1_con - 0.632:+.3f}')
